In [ ]:
import torch
import numpy as np
import cv2
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


config = {
    'dataset': {'image_size': 384},
    'model': {'filters': 48}
}

class ImprovedDepthModel(nn.Module):
    def __init__(self):
        super(ImprovedDepthModel, self).__init__()
        
        # Encoder
        self.down1 = nn.Sequential(
            nn.Conv2d(3, config['model']['filters'], kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(config['model']['filters']),
            nn.ReLU(inplace=True)
        )
        self.pool1 = nn.MaxPool2d(2)
        
        self.down2 = nn.Sequential(
            nn.Conv2d(config['model']['filters'], config['model']['filters']*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*2),
            nn.ReLU(inplace=True)
        )
        self.pool2 = nn.MaxPool2d(2)
        
        self.down3 = nn.Sequential(
            nn.Conv2d(config['model']['filters']*2, config['model']['filters']*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*4),
            nn.ReLU(inplace=True)
        )
        self.pool3 = nn.MaxPool2d(2)
        
        self.down4 = nn.Sequential(
            nn.Conv2d(config['model']['filters']*4, config['model']['filters']*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*8),
            nn.ReLU(inplace=True)
        )
        self.pool4 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(config['model']['filters']*8, config['model']['filters']*16, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*16),
            nn.ReLU(inplace=True)
        )

        # Decoder
        self.up4 = nn.ConvTranspose2d(config['model']['filters']*16, config['model']['filters']*8, kernel_size=4, stride=2, padding=1)
        self.conv4 = nn.Sequential(
            nn.Conv2d(config['model']['filters']*16, config['model']['filters']*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*8),
            nn.ReLU(inplace=True)
        )
        
        self.up3 = nn.ConvTranspose2d(config['model']['filters']*8, config['model']['filters']*4, kernel_size=4, stride=2, padding=1)
        self.conv3 = nn.Sequential(
            nn.Conv2d(config['model']['filters']*8, config['model']['filters']*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*4),
            nn.ReLU(inplace=True)
        )
        
        self.up2 = nn.ConvTranspose2d(config['model']['filters']*4, config['model']['filters']*2, kernel_size=4, stride=2, padding=1)
        self.conv2 = nn.Sequential(
            nn.Conv2d(config['model']['filters']*4, config['model']['filters']*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*2),
            nn.ReLU(inplace=True)
        )
        
        self.up1 = nn.ConvTranspose2d(config['model']['filters']*2, config['model']['filters'], kernel_size=4, stride=2, padding=1)
        self.conv1 = nn.Sequential(
            nn.Conv2d(config['model']['filters']*2, config['model']['filters'], kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']),
            nn.ReLU(inplace=True)
        )
        
        # Output layer
        self.output = nn.Sequential(
            nn.Conv2d(config['model']['filters'], config['model']['filters']//2, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(config['model']['filters']//2, 1, kernel_size=1),
            nn.ReLU()
        )
    
    def forward(self, x):
        x = F.interpolate(x, size=(config['dataset']['image_size'], config['dataset']['image_size']), 
                         mode='bilinear', align_corners=True)

        # Encoder path
        d1 = self.down1(x)
        p1 = self.pool1(d1)
        
        d2 = self.down2(p1)
        p2 = self.pool2(d2)
        
        d3 = self.down3(p2)
        p3 = self.pool3(d3)
        
        d4 = self.down4(p3)
        p4 = self.pool4(d4)
        
        # Bottleneck
        b = self.bottleneck(p4)

        u4 = self.up4(b)
        u4 = torch.cat([u4, d4], dim=1)
        c4 = self.conv4(u4)
        
        u3 = self.up3(c4)
        u3 = torch.cat([u3, d3], dim=1)
        c3 = self.conv3(u3)
        
        u2 = self.up2(c3)
        u2 = torch.cat([u2, d2], dim=1)
        c2 = self.conv2(u2)
        
        u1 = self.up1(c2)
        u1 = torch.cat([u1, d1], dim=1)
        c1 = self.conv1(u1)
        
        # Final output
        out = self.output(c1)
        
        return out

def load_model(model_path='best_depth_model.pth'):
    model = ImprovedDepthModel().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    return model

def process_frame(model, frame):
    img_size = config['dataset']['image_size']
    
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    h, w, _ = frame_rgb.shape
    
    img_tensor = torch.from_numpy(frame_rgb).float().permute(2, 0, 1) / 255.0
    img_tensor = img_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        depth = model(img_tensor)

    depth = depth.squeeze().cpu().numpy()
    depth = (depth - depth.min()) / (depth.max() - depth.min())

    depth_resized = cv2.resize(depth, (w, h), interpolation=cv2.INTER_CUBIC)

    depth_colored = cv2.applyColorMap((depth_resized * 255).astype(np.uint8), cv2.COLORMAP_MAGMA)
    
    return depth_colored

def process_video(model, video_path, output_path=None):
    if not os.path.exists(video_path):
        print(f"Video file not found: {video_path}")
        return
    
    if output_path is None:

        filename = os.path.basename(video_path)
        name, ext = os.path.splitext(filename)
        output_path = f"{name}_depth{ext}"

    cap = cv2.VideoCapture(video_path)

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
  
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    print(f"Processing video with {frame_count} frames...")
    for _ in tqdm(range(frame_count)):
        ret, frame = cap.read()
        if not ret:
            break

        depth_frame = process_frame(model, frame)

        out.write(depth_frame)

    cap.release()
    out.release()
    
    print(f"Done! Output saved to {output_path}")

if __name__ == "__main__":

    video_path = input("Enter path to video file (.mp4): ")
    
    print("Loading model...")
    model = load_model()

    process_video(model, video_path)